In [1]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from rapidfuzz import process, fuzz
import re
import unicodedata
from utils.fetch_data import fetch_soil, fetch_weather

In [2]:
def normalize_text(x):
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x


def fuzzy_match_district(df, centroid, threshold=85):
    # Normalize both sides
    df["District_norm"] = df["District"].apply(normalize_text)
    centroid["District_norm"] = centroid["District"].apply(normalize_text)
    centroid["State_norm"] = centroid["State"].apply(normalize_text)
    df["State_norm"] = df["State"].apply(normalize_text)

    # Drop duplicates in centroid to prevent merge explosion
    centroid = centroid.drop_duplicates(subset=["State_norm", "District_norm"])

    # Build fast lookup per state
    state_to_districts = (
        centroid.groupby("State_norm")["District_norm"].apply(list).to_dict()
    )

    def match_row(row):
        dist = row["District_norm"]
        state = row["State_norm"]

        if state in state_to_districts and dist in state_to_districts[state]:
            return dist

        # Try fuzzy match in same state first
        if state in state_to_districts:
            match, score, _ = process.extractOne(
                dist, state_to_districts[state], scorer=fuzz.WRatio
            )  # type: ignore
            if score >= threshold:
                return match

        # If still not found, fuzzy match across all districts
        all_dists = centroid["District_norm"].tolist()
        match, score, _ = process.extractOne(dist, all_dists, scorer=fuzz.WRatio)  # type: ignore
        if score >= threshold:
            return match

        return dist

    df["District_norm"] = df.apply(match_row, axis=1)

    merged = df.merge(
        centroid[["State_norm", "District_norm", "Latitude", "Longitude"]].rename(
            columns={
                "Latitude": "latitude",
                "Longitude": "longitude",
            }
        ),
        on=["State_norm", "District_norm"],
        how="left",
    )

    return merged

In [3]:
import os

df_new = pd.read_csv("data/converted_crop_data.csv")
df_new["Start_Year"] = df_new["Year"].apply(lambda x: int(x.split("-")[0]))

if os.path.exists("data/enriched_crop_data.csv"):
    df_existing = pd.read_csv("data/enriched_crop_data.csv")
    # Create a key for merging/filtering
    keys = ["State", "District", "Season", "Year", "Crop"]

    # Perform an anti-join to find rows in df_new that are not in df_existing
    merged_check = df_new.merge(df_existing[keys], on=keys, how="left", indicator=True)
    df_to_process = merged_check[merged_check["_merge"] == "left_only"].drop(
        columns=["_merge"]
    )
else:
    df_existing = pd.DataFrame()
    df_to_process = df_new

print(f"Total rows: {len(df_new)}")
print(f"Existing rows: {len(df_existing)}")
print(f"Rows to process: {len(df_to_process)}")

if not df_to_process.empty:
    centroid = pd.read_csv("data/centroid.csv")
    df_merged = fuzzy_match_district(df_to_process, centroid)
    df_merged.dropna(subset=["latitude", "longitude"], inplace=True)
    df_merged.info()
else:
    df_merged = pd.DataFrame()
    print("No new data to process.")

Total rows: 143418
Existing rows: 58953
Rows to process: 84465
<class 'pandas.core.frame.DataFrame'>
Index: 71990 entries, 0 to 83196
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   State          71990 non-null  object 
 1   District       71990 non-null  object 
 2   Season         71990 non-null  object 
 3   Year           71990 non-null  object 
 4   Crop           71990 non-null  object 
 5   Area_Ha        71990 non-null  float64
 6   Yield_QHa      71990 non-null  float64
 7   Start_Year     71990 non-null  int64  
 8   District_norm  71990 non-null  object 
 9   State_norm     71990 non-null  object 
 10  latitude       71990 non-null  float64
 11  longitude      71990 non-null  float64
dtypes: float64(4), int64(1), object(7)
memory usage: 7.1+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 71990 entries, 0 to 83196
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  --

In [4]:
def make_batches(df, batch_size=50):
    return np.array_split(df, max(1, len(df) // batch_size))


def append_data(df_merged):
    soil_keys = df_merged[
        ["State", "District", "latitude", "longitude"]
    ].drop_duplicates()
    soil_batches = make_batches(soil_keys, batch_size=50)

    soil_records = []
    print(f"Fetching soil data in {len(soil_batches)} batches...")

    for batch in tqdm(soil_batches):
        for _, row in batch.iterrows():
            data = fetch_soil(row["latitude"], row["longitude"])
            data["State"] = row["State"]
            data["District"] = row["District"]
            soil_records.append(data)

        time.sleep(0.5)  # polite delay between batches

    soil_df = pd.DataFrame(soil_records)
    df_merged = df_merged.merge(soil_df, on=["State", "District"], how="left")

    weather_keys = df_merged[
        ["State", "District", "latitude", "longitude", "Season", "Start_Year"]
    ].drop_duplicates()

    weather_batches = make_batches(weather_keys, batch_size=50)

    weather_records = []
    print(f"Fetching weather data in {len(weather_batches)} batches...")

    for batch in tqdm(weather_batches):
        for _, row in batch.iterrows():
            data = fetch_weather(
                row["latitude"], row["longitude"], row["Start_Year"], row["Season"]
            )
            data["State"] = row["State"]
            data["District"] = row["District"]
            data["Season"] = row["Season"]
            data["Start_Year"] = row["Start_Year"]
            weather_records.append(data)

        time.sleep(0.5)

    weather_df = pd.DataFrame(weather_records)

    df_merged = df_merged.merge(
        weather_df, on=["State", "District", "Season", "Start_Year"], how="left"
    )

    return df_merged

In [ ]:
if not df_merged.empty:
    df_processed = append_data(df_merged)
    if not df_existing.empty:
        df_final = pd.concat([df_existing, df_processed], ignore_index=True)
    else:
        df_final = df_processed
else:
    df_final = df_existing

# Drop duplicates based on key columns
if not df_final.empty:
    df_final.drop_duplicates(
        subset=["State", "District", "Season", "Year", "Crop"], inplace=True
    )

df_final

e:\Projects\SBS_HackTheGap_2026\python(ML)\sbs\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Fetching soil data in 6 batches...


  0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
df_final.info()

In [ ]:
df_final.to_csv("data/enriched_crop_data.csv", index=False)